# Homework: Vector Search

## Question 1: Embedding a query

> Embed the following query:
>
>    How does approximate nearest neighbor search work?
>
> The embedder returns a vector of 384 numbers. What's the first value (v[0])?
>
>    * -0.31
>    * -0.02
>    * 0.12
>    * 0.44

In [1]:
from embedder import Embedder

embed = Embedder()

q = 'How does approximate nearest neighbor search work?'
v = embed.encode(q)

print(v[0])

2026-06-24 22:58:23.036472760 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


-0.02058203437252893


## Question 2: Cosine similarity

> The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.
> 
> Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?
>
>    * 0.07
>    * 0.37
>    * 0.68
>    * 0.92

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

doc = None
for d in documents:
    if d['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md':
        doc = d['content']
        break

vector = embed.encode(doc)
sim = vector.dot(v)
print (sim)


0.36107027225589694


## Question 3: Chunking and search by hand

> A full page covers several topics, which waters down its embedding.
> 
> We chunk the pages the same way as in homework 1:
> 
> from gitsource import chunk_documents
> chunks = chunk_documents(documents, size=2000, step=1000)
> 
> We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:
> 
> scores = X.dot(v)
> 
> Which file does the highest-scoring chunk belong to (its filename)?
> 
>    * 02-vector-search/lessons/03-embeddings-dataset.md
>    * 02-vector-search/lessons/06-rag-vector.md
>    * 02-vector-search/lessons/07-sqlitesearch-vector.md
>    * 02-vector-search/lessons/09-onnx-embedder.md


In [3]:
from gitsource import chunk_documents
import numpy as np

chunks = chunk_documents(documents, size=2000, step=1000)
text_content = [chunk['content'] for chunk in chunks]

X = embed.encode_batch(text_content)
print(X.shape)

X = np.array(X)

scores = X.dot(v)
idx = np.argmax(scores)
print("Score {} from file: {}".format(scores[idx], chunks[idx]["filename"]))


(295, 384)
Score 0.6489017718578813 from file: 02-vector-search/lessons/07-sqlitesearch-vector.md


## Question 4: Vector search with minsearch
>
> We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.
> 
> Let's use VectorSearch from minsearch and run a search for the following query:
> 
>>     What metric do we use to evaluate a search engine?
> 
> Which file is the filename of the first result?
> 
>    * 02-vector-search/lessons/04-vector-search.md
>    * 04-evaluation/lessons/05-search-metrics.md
>    * 04-evaluation/lessons/13-llm-as-judge.md
>    * 05-monitoring/lessons/04-metrics.md


In [ ]:
from minsearch import VectorSearch

vectors = embed.encode_batch([d['content'] for d in documents])
vindex = VectorSearch(keyword_fields=[])
# vindex.fit(vectors, documents)
vindex.fit(X, chunks)
q4_query = 'What metric do we use to evaluate a search engine?'
q4_query_vector = embed.encode(q4_query)

q4_results = vindex.search(q4_query_vector, num_results=1)
print(q4_results[0]['filename'])

04-evaluation/lessons/05-search-metrics.md


## Question 5: Text search vs vector search

> Vector search matches by meaning, keyword search by exact words.
> 
> Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.
> 
> Run both searches for this query:
> 
>>     How do I store vectors in PostgreSQL?
> 
> Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?
> 
>    * 02-vector-search/lessons/01-intro.md
>    * 02-vector-search/lessons/02-embeddings.md
>    * 02-vector-search/lessons/08-pgvector.md
>    * 03-orchestration/lessons/05-rag.md

In [15]:
q5_query = 'How do I store vectors in PostgreSQL?'

from minsearch import Index

q5_index = Index(
    text_fields=["content"],
    keyword_fields=[]
)

q5_index.fit(chunks)
q5_keyword_result = q5_index.search(q5_query, num_results=5)
for r in q5_keyword_result:
    print('Keyword result: {}'.format(r['filename']))

q5_query_vector = embed.encode(q5_query)
q5_vector_result = vindex.search(q5_query_vector, num_results=5)
for r in q5_vector_result:
    print('Vector result: {}'.format(r['filename']))



Keyword result: 02-vector-search/lessons/02-embeddings.md
Keyword result: 03-orchestration/lessons/05-rag.md
Keyword result: 02-vector-search/lessons/01-intro.md
Keyword result: 03-orchestration/lessons/05-rag.md
Keyword result: 02-vector-search/lessons/01-intro.md
Vector result: 02-vector-search/lessons/08-pgvector.md
Vector result: 02-vector-search/lessons/08-pgvector.md
Vector result: 03-orchestration/lessons/05-rag.md
Vector result: 02-vector-search/lessons/08-pgvector.md
Vector result: 02-vector-search/lessons/08-pgvector.md


## Question 6: Hybrid search
